# Project: The Data Analyst Agent

**What you will build:** a capstone that combines everything — an agent that answers questions about a dataset by **writing and running its own Python** (a code-execution tool), with **memory**, a **guardrail**, and a small **eval**. It's the code version of the n8n "Ask Your Data" project.

One general "run code" tool beats a hundred narrow ones — the same idea behind coding agents like Claude Code.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/32_project_data_analyst.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
model = OpenAIChatModel(MODEL_NAME, provider=OpenAIProvider(
    base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"]))
print("Ready:", MODEL_NAME)
import pandas as pd, io, contextlib

## The data

A tiny sales dataset. In a real project this would be your CSV, a database query, or an uploaded file.

In [ ]:
CSV = """month,product,units,revenue
Jan,Widget,120,2400
Jan,Gadget,80,3200
Feb,Widget,150,3000
Feb,Gadget,95,3800
Mar,Widget,90,1800
Mar,Gadget,130,5200"""
df = pd.read_csv(io.StringIO(CSV))
df

## The code-execution tool

Instead of writing one tool per question (`total_revenue()`, `best_month()`, ...), we give the agent a single tool that runs Python against the dataframe. The agent writes the analysis itself. This is *programmatic tool use*: one general "run code" tool instead of a hundred narrow ones.

In [ ]:
def run_python(code: str) -> str:
    """Run Python to analyze the sales dataframe `df` (pandas is `pd`). print() the answer."""
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {"df": df, "pd": pd})
        return buf.getvalue().strip() or "(ran, but nothing was printed — remember to print() the result)"
    except Exception as e:
        return f"Error: {e}"   # feed the error back so the agent can fix its own code (0.3, 19)

print(run_python("print(df['revenue'].sum())"))   # quick sanity check

```{warning}
`exec` runs arbitrary code — fine for *your own* analysis in a notebook, but **never expose this to untrusted users as-is**. In production you'd sandbox it, in rough order of strength: a locked-down `subprocess` with a timeout, a container, or a hosted code-interpreter service (e.g. E2B). We keep it simple here to teach the pattern; the n8n course makes the same trade-off and says so.
```

## The agent (the system prompt *is* the engineering)

The workflow is three lines; the quality lives in the system prompt. We tell the agent the schema, that it must use the tool, and to answer only from what the code prints — the same iterative prompt-engineering spine as n8n's Ask Your Data.

In [ ]:
SCHEMA = "The dataframe `df` has columns: month (Jan/Feb/Mar), product (Widget/Gadget), units (int), revenue (USD int)."

analyst = Agent(
    model,
    system_prompt=(
        "You are a data analyst. Answer questions about the sales data by writing Python and calling run_python. "
        f"{SCHEMA} Always print() the result inside the code. Base your answer ONLY on what the tool prints, "
        "and state the numbers you found."
    ),
)
analyst.tool_plain(run_python)   # register the function defined above as a tool

r = await analyst.run("Which product made more total revenue, and by how much?")
print(r.output)

The agent wrote pandas code, ran it, read the printed numbers, and answered from them. Now let's layer on the rest of the course.

## Add memory: follow-up questions

Pass `message_history` (1.4) so the agent handles follow-ups that depend on the previous turn.

In [ ]:
r1 = await analyst.run("What was total revenue in February?")
print("Q1:", r1.output)

r2 = await analyst.run("And how does that compare to January?", message_history=r1.all_messages())
print("Q2:", r2.output)   # 'that' resolves via memory

## Add a guardrail: stay on task

A data agent shouldn't answer off-topic questions (or run unrelated code). A quick input screen (1.5) keeps it scoped.

In [ ]:
async def ask_analyst(question, history=None):
    off_topic = ["weather", "joke", "who are you", "password"]
    if any(w in question.lower() for w in off_topic):
        return "I only answer questions about the sales dataset."
    r = await analyst.run(question, message_history=history or [])
    return r.output

print(await ask_analyst("Tell me a joke"))                      # refused
print(await ask_analyst("Which month had the highest revenue?"))  # answered

## Add an eval: is it actually right?

Non-deterministic + writing its own code = you must **measure** (1.6). We compute ground truth with pandas and check the agent's answer contains the right number, over a few cases.

In [ ]:
cases = [
    ("What is the total revenue across all months?", str(df['revenue'].sum())),
    ("How many Widget units were sold in total?",     str(df[df['product']=='Widget']['units'].sum())),
    ("Which product sold more total units?",           "Gadget" if df[df.product=='Gadget'].units.sum() > df[df.product=='Widget'].units.sum() else "Widget"),
]

passed = 0
for question, truth in cases:
    answer = await ask_analyst(question)
    ok = truth.lower() in answer.lower()
    passed += ok
    print(f"[{'PASS' if ok else 'FAIL'}] expected {truth!r} in -> {answer[:80]}")
print(f"\nScore: {passed}/{len(cases)}")

That score is your ground truth. Change the model or the system prompt, re-run this cell, and you *know* whether it got better — the whole point of evals. **You just combined tools + memory + guardrails + evals into one agent.** That's a portfolio project.

::::{dropdown} 🛠️ Extend it (challenges)
:color: secondary

1. **Use your own data.** Replace `CSV` with a real dataset (load a file in Colab) and update `SCHEMA`. Watch how much the schema in the prompt matters.
2. **Add a chart tool.** Give the agent a second tool that runs matplotlib and saves a PNG; ask it to plot revenue by month.
3. **Harden the guardrail.** Move the off-topic check into an `@analyst.output_validator` (1.5) and also block code that imports modules.
4. **Grow the eval.** Add five more cases with known answers and track your pass rate as you tweak the prompt.
5. **Ship it.** Wrap this agent in the FastAPI service from 30 and call it from a front end (31).
::::

---

**That's the course.** From a raw `while` loop (0.3) to a deployed, evaluated, memory-and-guardrail agent that writes its own code. You understand every layer, because you built each one before adopting the framework that automates it. See **33** for the map of which tool to reach for next time — and then go build something real.